In [ ]:
import packages.package1.test as pk1

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import liftover

In [ ]:
genelist = list(pd.read_csv("data/ddg2p.genelist.v2.tsv", sep='\t', header=None, names=["genename"])["genename"])
len(genelist)

In [ ]:
ddg2p_genelist_df = pd.read_csv("data/ddg2p_genelist_mutability_df.tsv", sep='\t', low_memory = False)

In [ ]:
ddg2p_genelist_df.head()

In [ ]:
ddg2p_genelist_df.columns

In [ ]:
database_location = "data/gnomad_gen_db.sqlite3"
gen_db = gnomAD_DB(database_location, gnomad_version="v4")

In [ ]:
import os
import pyliftover
from pyliftover import LiftOver
lo = LiftOver('hg19', 'hg38')

In [ ]:
res = lo.convert_coordinate("chr10", 12111033)
res

In [ ]:
gene_chr = {}
gene_pos_min = {}
gene_pos_max = {}

i = 0
for gene in genelist:
    gene_df = ddg2p_genelist_df[ddg2p_genelist_df["genename"] == gene]
    i+=1
    if i%100 == 0:
        print(i)
    if len(gene_df[~gene_df['hg19_chr'].isna()]['hg19_chr'])>0:
        chrom = "chr" + gene_df[~gene_df['hg19_chr'].isna()]['hg19_chr'].unique()

        gene_chr[gene] = gene_df[~gene_df['hg19_chr'].isna()]['hg19_chr'].unique()[0]
        min_pos = gene_df['hg19_pos(1-based)'].min()
        max_pos = gene_df['hg19_pos(1-based)'].max()

        gene_pos_min[gene] = list(lo.convert_coordinate(chrom[0], min_pos)[0])[1]
        gene_pos_max[gene] = list(lo.convert_coordinate(chrom[0], max_pos)[0])[1]
    else:
        print(gene)
    

In [ ]:
df_gene_chr = pd.DataFrame([gene_chr])
df_gene_chr.to_csv('data/gene_chr.csv')
df_gene_pos_min = pd.DataFrame([gene_pos_min])
df_gene_pos_min.to_csv('data/gene_pos_min.csv')
df_gene_pos_max = pd.DataFrame([gene_pos_max])
df_gene_pos_max.to_csv('data/gene_pos_max.csv')

# Mutational Score

In [ ]:
codons = pd.read_csv("data/codons_stat_human_genscript.tsv.csv", sep='\t', low_memory=False, na_values=['.'])
is_stop = codons["Amino acid"] == "*"

In [ ]:
def hamming_distance(chaine1, chaine2):
    return sum(c1 != c2 for c1, c2 in zip(chaine1, chaine2))

def to_3_letters(aa):
    return codons[codons["Amino acid"]==aa]["Residue"].values[0]

def mutations(triplet, aa):
    return [[triplet, aa, x, codons[codons["Triplet"]==x]["Amino acid"].values[0]] \
            for x in list(codons[~is_stop & (codons["Amino acid"] != aa)]["Triplet"]) \
            if hamming_distance(triplet, x) == 1]

def mutability(triplet, aa):
    return len([x for x in list(codons[~is_stop & (codons["Amino acid"] != aa)]["Triplet"]) if hamming_distance(triplet, x) == 1])

In [ ]:
codons["Mutability"] = codons.apply(lambda row: mutability(row["Triplet"], row["Amino acid"]), axis=1)

In [ ]:
mut_hash = codons.groupby('Triplet')['Mutability'].apply(list).to_dict()

In [ ]:
codon_mutability_table = []
for n1 in "TCAG":
    for n3 in "TCAG":
        codon_mutability_table.append(["{} {}".format(codons[codons["Triplet"]==n1+n2+n3]["Residue"].values[0]\
                                                        , mut_hash[n1+n2+n3]) for n2 in "TCAG"])

In [ ]:
mut_hash = codons.groupby('Triplet')['Mutability'].apply(list).to_dict()
mut_hash_aa = codons.groupby('Amino acid')['Mutability'].mean().to_dict()

# Loading GnomAD 4 Data

In [ ]:
import pandas as pd
from gnomad_db.database import gnomAD_DB

In [ ]:
gene_chr = pd.read_csv('data/gene_chr.csv')
del gene_chr["Unnamed: 0"]
gene_pos_min = pd.read_csv('data/gene_pos_min.csv')
del gene_pos_min["Unnamed: 0"]
gene_pos_max = pd.read_csv('data/gene_pos_max.csv')
del gene_pos_max["Unnamed: 0"]

In [ ]:
import os
import pyliftover
from pyliftover import LiftOver
lo = LiftOver('hg19', 'hg38')

## Loading Exomes

In [ ]:
database_location = "data/gnomad_gen_db.sqlite3"
ex_db = gnomAD_DB(database_location, gnomad_version="v4")

In [ ]:
gnomad4_ex_df = ex_db.get_info_for_interval(chrom=gene_chr['HMX1'][0], interval_start=gene_pos_min['HMX1'][0], 
                                      interval_end=gene_pos_max['HMX1'][0], query="*")

# ADD GENE INFORMATION FOR LATER ON

In [ ]:
i = 0
for gene in gene_chr.columns:
    i = i + 1
    if i%100 == 0:
        break
    temp_df = ex_db.get_info_for_interval(chrom=gene_chr[gene][0], interval_start=gene_pos_min[gene][0], 
                                      interval_end=gene_pos_max[gene][0], query="*")
    gnomad4_ex_df = pd.concat([temp_df, gnomad4_ex_df])

In [ ]:
gnomad4_ex_df.to_csv("data/gnomad4_ex_df.csv")

In [ ]:
gnomad4_ex_df.columns

In [ ]:
gnomad4_ex_df.head()[['AN', 'AF',
       'MQ', 'QD', 'ReadPosRankSum']]

# Loading Genomes

In [ ]:
database_location = "data/gnomad_gen_db.sqlite3"
gen_db = gnomAD_DB(database_location, gnomad_version="v4")

In [ ]:
gnomad4_gen_df = gen_db.get_info_for_interval(chrom=gene_chr['HMX1'][0], interval_start=gene_pos_min['HMX1'][0], 
                                      interval_end=gene_pos_max['HMX1'][0], query="*")

In [ ]:
for gene in gene_chr.columns:
    temp_df = gen_db.get_info_for_interval(chrom=gene_chr[gene][0], interval_start=gene_pos_min[gene][0], 
                                      interval_end=gene_pos_max[gene][0], query="*")
    gnomad4_gen_df = pd.concat([temp_df, gnomad4_gen_df])

In [ ]:
gnomad4_gen_df.to_csv("data/gnomad4_gen_df.csv")

# Bundling Genomes and Exomes

In [ ]:
gnomad4_ex_df = pd.read_csv("data/gnomad4_ex_df.csv")
gnomad4_gen_df = pd.read_csv("data/gnomad4_gen_df.csv")

In [ ]:
gnomad4_gen_df = gnomad4_gen_df[['chrom', 'pos', 'ref', 'alt', 'AC', 'AN', 'AF']]
gnomad4_ex_df = gnomad4_ex_df[['chrom', 'pos', 'ref', 'alt', 'AC', 'AN', 'AF']]

In [ ]:
gnomad4_df = pd.merge(gnomad4_gen_df, gnomad4_ex_df, on=['chrom', 'pos', 'ref', 'alt'],
                                suffixes = ("_gnomAD_exomes","_gnomAD_genomes"), how = 'outer')

In [ ]:
gnomad4_df

In [ ]:
ddg2p_genelist_df.columns

In [ ]:
test = ddg2p_genelist_df[['#chr', 'pos(1-based)', 'aaref', 'aaalt', 'refcodon', 'codonpos', 'genename']]
test.columns =  ['chrom', 'pos', 'aaref', 'aaalt', 'refcodon', 'codonpos', "genename"]

In [ ]:
gnomad4_df["chrom"] = gnomad4_df["chrom"].astype(str)

In [ ]:
gn_test = pd.merge(gnomad4_df, test, on=['chrom', 'pos'], how = 'inner')

In [ ]:
gn_test[gn_test["genename"] == "FBN1"]

# Applying the pipeline

In [ ]:
def sum_mut(gene):
    return gn_test[~alt_is_stop & (gn_test["genename"] == gene)].groupby("aaref").size().to_dict()
    
def sum_mut_gnomad(gene):
    return gn_test[~alt_is_stop & (gn_test["genename"] == gene) & \
                             (gencond | exocond)].groupby("aaref").size().to_dict()

In [ ]:
1/500000

In [ ]:
gencond = gn_test["AF_gnomAD_exomes"] > 0.00005
exocond = gn_test["AF_gnomAD_exomes"] > 0.00005

In [ ]:
alt_is_stop = gn_test["aaalt"] == 'X'

In [ ]:
gn_test = gn_test[gn_test["aaref"] != "X"]

In [ ]:
rate_aa_gnomad = dict()
for aa in "ARNDCQEGHILKMFPSTWYV":
    rate_aa_gnomad[to_3_letters(aa)] = 1.0*len(gn_test[(exocond | gencond) & \
                                         (gn_test["ref"] == aa)])/len(gn_test[(exocond | gencond)]) 

In [ ]:
rate_aa_gnomad

In [ ]:
for aa in "ARNDCQEGHILKMFPSTWYV":
    print("Amino acid triplet distribution for "+ to_3_letters(aa))
    print(gn_test[gn_test["refcodon"].isin(list(codons[~is_stop & (codons["Amino acid"]==aa)]["Triplet"]))]["refcodon"].value_counts())

In [ ]:
mut_hash = codons.groupby('Triplet')['Mutability'].apply(list).to_dict()

In [ ]:
gn_test.to_csv("gnomad4_ex_gen_df_00001.csv")

In [ ]:
gn_test.shape

In [ ]:
gn_test["mutability"] = gn_test.apply(lambda row: \
                                        mut_hash[row["refcodon"]][0]\
                                        if row["refcodon"] in list(codons["Triplet"]) else None, axis=1)

In [ ]:
sum_mut_all = dict()
for aa in "ARNDCQEGHILKMFPSTWYV":
    sum_mut_all[aa] = gn_test[gn_test["aaref"]==aa]["mutability"].sum()

In [ ]:
sum_mut_all

In [ ]:
sum_mut_all_gnomad = dict()
for aa in "ARNDCQEGHILKMFPSTWYV":
    sum_mut_all_gnomad[aa] = gn_test[(gencond | exocond) & (gn_test["aaref"]==aa)]["mutability"].sum()
sum_mut_all_gnomad

In [ ]:
sum_var_all_gnomad = dict()
for aa in "ARNDCQEGHILKMFPSTWYV":
    sum_var_all_gnomad[aa] = len(gn_test[(gencond | exocond) & (gn_test["aaref"]==aa)])

In [ ]:
ratio = dict()
for k, v in sum_mut_all.items():
    ratio[k] = 100.0*sum_var_all_gnomad[k]/sum_mut_all[k]

In [ ]:
import operator
ratio_s = sorted(ratio.items(), key=operator.itemgetter(1))
ratio_s

In [ ]:
def build_ratios(gene):
    sum_mut_gene = sum_mut(gene)
    sum_mut_gene_gnomad = sum_mut_gnomad(gene)
    ratio = dict()
    for aa in "ARNDCQEGHILKMFPSTWYV":
        if aa not in sum_mut_gene.keys():
            ratio[aa] = None
        elif aa not in sum_mut_gene_gnomad.keys():
            ratio[aa] = 0
        else:
            ratio[aa] = 100.0*sum_mut_gene_gnomad[aa]/sum_mut_gene[aa]
    return ratio

In [ ]:
len(genelist)

In [ ]:
i=0
ratio_list = {}
print(len(genelist))
for gene in genelist:
    i+=1
    print(i)
    ratio = build_ratios(gene)
    ratio_list[gene] = ratio

In [ ]:
ratio_list_df = pd.DataFrame(ratio_list)

In [ ]:
ratio_df = pd.DataFrame.from_dict(ratio_list, orient="index")

In [ ]:
ratio_list_df.to_csv("data/ratio_list_gnomad4_00005.csv")

In [ ]:
ratio_list_df = pd.read_csv("data/ratio_list_gnomad4_00005.csv")
ratio_list_df = ratio_list_df.set_index("Unnamed: 0")
ratio_list_df

In [ ]:
ratio_list_df["FBN1"]

In [ ]:
import numpy as np
ratio_list_test = np.transpose(ratio_list_df)

In [ ]:
ratio_list_test.mean()

In [ ]:
margin_table = dict()
for gene in ratio_df.index:
    margin_table[gene] = dict()
    for aa in "ARNDCQEGHILKMFPSTWYV":
        if ratio_df.loc[gene,aa] is not None:
            margin_table[gene][aa] = (ratio_df.loc[gene,aa]-ratio_df[aa].mean())/ratio_df[aa].std()
        else:
            margin_table[gene][aa] = None

In [ ]:
margin_df = pd.DataFrame.from_dict(margin_table, orient="index")

In [ ]:
def gene_stats(gene, aa, file):
    test = list(ratio_list_df.loc[aa])
    res = []
    for val in test:
        if val != None :
            res.append(val)
    max_hist = max(res)
    gene_value = ratio_list_df.loc[aa,gene]
    
    fig, ax = plt.subplots(figsize = (10,10))

    plt.hist(x=list(res), bins=range(0,int(max_hist+1),2))

    plt.plot([ratio_list_df.loc[aa, gene], ratio_list_df.loc[aa, gene]],[0,450], color = "r", marker = 'o')
    plt.plot([ratio_list_df.mean()[gene], ratio_list_df.mean()[gene]],[0,450], color = "b", marker = 'o')

    plt.savefig('output/' + file)
    
    plt.show()


In [ ]:
def get_cmap(n, name='hsv'):
    '''Returns a function that maps each index in 0, 1, ..., n-1 to a distinct 
    RGB color; the keyword argument name must be a standard mpl colormap name.'''
    return plt.cm.get_cmap(name, n)

In [ ]:
ratio_list_df

In [ ]:
figure, axis = plt.subplots(7, 3, figsize = (20,30))

gene = "FBN1"

num_1 = 0
num_2 = 0
i = 0
for aa in "ARNDCQEGHILKMFPSTWYV":
    i = i + 1
    test = list(ratio_list_df.loc[aa])
    res = []
    for val in test:
        if val != None :
            res.append(val)
    max_hist = max(res)
    gene_value = ratio_list_df.loc[aa,gene]
    axis[num_2, num_1].figsize = (10,10)
    axis[num_2, num_1].hist(x=list(res), bins=range(0,int(max_hist+1),2), color = "lightsteelblue")
    axis[num_2, num_1].hist(x=list(res), color = "lightsteelblue")

    axis[num_2, num_1].plot([ratio_list_df.loc[aa, gene], ratio_list_df.loc[aa, gene]],[0,450], color = "k", marker = 'o')
    axis[num_2, num_1].plot([ratio_list_df.loc[aa].mean(), ratio_list_df.loc[aa].mean()],[0,450], color = "b", marker = 'o')
    axis[num_2, num_1].set_title("Amino Acid " + aa)
    num_1 += 1
    
    if num_1 > 2:
        num_1 = 0
        num_2 += 1
plt.savefig('output/' + "Amino_Acid_View_test_gnomad4.png")
plt.show()


In [ ]:
figure, axis = plt.subplots(7, 3, figsize = (20,30))

num_1 = 0
num_2 = 0
i = 0
for aa in "ARNDCQEGHILKMFPSTWYV":
    i = i + 1
    test = list(ratio_list_df.loc[aa])
    res = []
    for val in test:
        if val != None :
            res.append(val)
    max_hist = max(res)
    y, x, _ = plt.hist(x=list(res), bins=[0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60])
    
    dist = 9999
    el = 0
    for a in x:
        if abs(ratio_list_df.loc[aa].mean() - a) < dist:
            if ratio_list_df.loc[aa].mean() - a >= 0:
                dist = ratio_list_df.loc[aa].mean() - a
                x_to = el
        el = el + 1
    mean_y = y[x_to]
    
    
    axis[num_2, num_1].figsize = (10,10)
    axis[num_2, num_1].hist(x=list(res), bins=[0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60], color = (0.1, 0.1, 0.9, 0.1), edgecolor = "lightgrey")

    #axis[num_2, num_1].plot([ratio_list_df.loc[aa, gene], ratio_list_df.loc[aa, gene]],[0,gene_y], color = "black", marker = 'o')
    axis[num_2, num_1].plot([ratio_list_df.loc[aa].mean(), ratio_list_df.loc[aa].mean()],[0,mean_y], color = "goldenrod", marker = 'o')
    axis[num_2, num_1].set_title("Amino Acid " + aa)
    num_1 += 1
    
    if num_1 > 2:
        num_1 = 0
        num_2 += 1
plt.savefig('output/' + "Amino_Acid_View_test_gnomAD4.png")
plt.show()

In [ ]:
figure, axis = plt.subplots(7, 3, figsize = (20,30))

gene = "FBN1"

num_1 = 0
num_2 = 0
i = 0
for aa in "ARNDCQEGHILKMFPSTWYV":
    i = i + 1
    test = list(ratio_list_df.loc[aa])
    res = []
    for val in test:
        if val != None :
            res.append(val)
    max_hist = max(res)
    gene_value = ratio_list_df.loc[aa,gene]
    
    y, x, _ = plt.hist(x=list(res), bins=list(range(0,35,5)))
    
    dist = 9999
    el = 0
    for a in x:
        if abs(ratio_list_df.loc[aa, gene] - a) < dist:
            if ratio_list_df.loc[aa, gene] - a >= 0:
                dist = ratio_list_df.loc[aa, gene] - a
                x_to = el
        el = el + 1

    gene_y = y[x_to]
    
    dist = 9999
    el = 0
    for a in x:
        if abs(ratio_list_df.loc[aa].mean() - a) < dist:
            if ratio_list_df.loc[aa].mean() - a >= 0:
                dist = ratio_list_df.loc[aa].mean() - a
                x_to = el
        el = el + 1

    mean_y = y[x_to]
    
    
    axis[num_2, num_1].figsize = (1,1)

    axis[num_2, num_1].hist(x=list(res), bins=list(range(0,35,5)), color = (0.1, 0.1, 0.9, 0.1), edgecolor = "lightgrey")
    if aa == "C":
        axis[num_2, num_1].plot([1, 1],[0,630], color = "black", marker = 'o')
    else:
        axis[num_2, num_1].plot([ratio_list_df.loc[aa, gene], ratio_list_df.loc[aa, gene]],[0,gene_y], color = "black", marker = 'o')
    axis[num_2, num_1].plot([ratio_list_df.loc[aa].mean(), ratio_list_df.loc[aa].mean()],[0,mean_y], color = "goldenrod", marker = 'o')
    axis[num_2, num_1].set_title("Amino Acid " + aa)
    num_1 += 1
    
    if num_1 > 2:
        num_1 = 0
        num_2 += 1
plt.savefig('output/' + "Amino_Acid_View_test_gnomad4.png")
plt.show()
